# Lab 16 — Multi-agent evaluation harness (reference solution)

Canonical implementation of [Lab 16: Multi-agent evaluation harness from scratch](../README.md).

Five trajectory metrics + two outcome metrics over a 15-trace hand-curated fixture set replayed from Labs 10/11/12. The solution is the harness; the parent lab walks you through building it.

> 📖 See [`solution/README.md`](./README.md) for the design choices flagged.
> ⏱ Run time: ~3 seconds. No LLM calls.


## Setup

Pure-Python + pandas + pydantic. No new dependencies.

In [ ]:
import json
import pathlib
import re
from collections import Counter
from typing import Literal

import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError


## Trace models + load

Pydantic `StrictModel` with `extra="forbid"`. The trace_set is a contract.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class TraceStep(StrictModel):
    node: str
    args: dict
    output: dict
    status: str = "ok"


class Trace(StrictModel):
    id: str
    source_lab: Literal["lab10", "lab11", "lab12"]
    task: str
    trajectory: list[TraceStep]
    final_answer: str
    expected_handoffs: list[str]
    expected_citations: list[str]
    category: Literal["happy_path", "tool_failure", "replan_needed",
                      "citation_drift", "step_cap_hit"]


TRACE_SET_PATH = pathlib.Path("../trace_set.jsonl")
assert TRACE_SET_PATH.exists(), f"Missing {TRACE_SET_PATH}; run from solution/."

traces: list[Trace] = []
with TRACE_SET_PATH.open() as f:
    for line in f:
        if line.strip():
            traces.append(Trace.model_validate(json.loads(line)))

print(f"Loaded {len(traces)} traces.")
print(f"Categories: {dict(Counter(t.category for t in traces))}")


## Trajectory metrics

Five from-scratch pure functions. Each returns float, None (when N/A for the source lab), or dict (when the metric has structured subscores).

In [ ]:
def handoff_success_rate(trace: Trace) -> float:
    """Fraction of inter-agent handoffs where upstream completed cleanly
    (status in {ok, step_cap}) AND downstream received non-empty args."""
    if len(trace.trajectory) < 2:
        return 1.0
    succ = 0
    for i in range(1, len(trace.trajectory)):
        prev, curr = trace.trajectory[i - 1], trace.trajectory[i]
        if prev.status in ("ok", "step_cap") and bool(curr.args):
            succ += 1
    return succ / (len(trace.trajectory) - 1)


def routing_accuracy(trace: Trace) -> float:
    """LCS-based comparison of actual node sequence vs expected_handoffs.
    Normalized by expected length — penalizes skipped agents more than extras."""
    actual = [s.node for s in trace.trajectory]
    expected = trace.expected_handoffs
    n, m = len(actual), len(expected)
    if n == 0 or m == 0:
        return 0.0
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            dp[i + 1][j + 1] = (dp[i][j] + 1) if actual[i] == expected[j] else max(dp[i][j + 1], dp[i + 1][j])
    return dp[n][m] / m


def plan_validity(trace: Trace) -> float | None:
    """1.0 if the final planner step's output has plan_valid=True. None for non-Lab-12."""
    if trace.source_lab != "lab12":
        return None
    planner_steps = [s for s in trace.trajectory if s.node == "planner"]
    return float(bool(planner_steps and planner_steps[-1].output.get("plan_valid", False)))


def plan_coverage(trace: Trace) -> float | None:
    """successful_executor_steps / final_plan_step_count. None for non-Lab-12."""
    if trace.source_lab != "lab12":
        return None
    planner_steps = [s for s in trace.trajectory if s.node == "planner"]
    if not planner_steps:
        return 0.0
    plan_steps = len(planner_steps[-1].output.get("plan", {}).get("steps", []))
    if plan_steps == 0:
        return 0.0
    succ = sum(1 for s in trace.trajectory if s.node == "executor" and s.status == "ok")
    return min(succ / plan_steps, 1.0)


def replan_rate(trace_set: list[Trace]) -> dict:
    """Set-level metric: fraction of Lab 12 traces with >1 planner step."""
    lab12 = [t for t in trace_set if t.source_lab == "lab12"]
    if not lab12:
        return {"rate": 0.0, "with_replans": 0, "total_lab12": 0, "mean_replans": 0.0}
    counts = [sum(1 for s in t.trajectory if s.node == "planner") for t in lab12]
    replans = [max(c - 1, 0) for c in counts]
    return {
        "rate": sum(1 for r in replans if r > 0) / len(lab12),
        "with_replans": sum(1 for r in replans if r > 0),
        "total_lab12": len(lab12),
        "mean_replans": sum(replans) / len(lab12),
    }


## Outcome metrics

Two from-scratch pure functions. Both with structured returns — citation_preservation returns a dict (preservation/hallucinated/dropped); groundedness returns a dict (fraction/total/grounded).

In [ ]:
_TRACKING_PARAMS = {"utm_source", "utm_medium", "utm_campaign", "utm_term",
                      "utm_content", "ref", "fbclid", "gclid"}


def _canonicalize_url(url: str) -> str:
    """Normalize URL: lowercase scheme/host, strip trailing slash, fragment, tracking params."""
    from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
    p = urlparse(url.strip())
    path = p.path.rstrip("/") if p.path != "/" else p.path
    qs = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=True)
          if k not in _TRACKING_PARAMS]
    return urlunparse((p.scheme.lower(), p.netloc.lower(), path, p.params,
                        urlencode(qs), ""))


_URL_RE = re.compile(r'https?://[^\s\]\)<>"]+')


def citation_preservation(trace: Trace) -> dict:
    """{preservation, hallucinated_count, dropped_count}."""
    expected = {_canonicalize_url(u) for u in trace.expected_citations}
    final = {_canonicalize_url(u) for u in _URL_RE.findall(trace.final_answer)}
    if not expected:
        return {"preservation": 1.0 if not final else 0.0,
                "hallucinated_count": len(final), "dropped_count": 0}
    return {"preservation": len(expected & final) / len(expected),
            "hallucinated_count": len(final - expected),
            "dropped_count": len(expected - final)}


_STOPWORDS = {"the", "and", "for", "with", "from", "have", "this", "that",
              "these", "those", "their", "into", "been", "will", "also",
              "more", "than", "such", "some", "most", "other", "which"}


def _extract_claims(answer: str) -> list[str]:
    """Sentence-split body (after stripping `[N]` citation lines). ≥10 chars filter."""
    body = " ".join(line for line in answer.split("\n") if not re.match(r"^\[\d+\]", line.strip()))
    return [s.strip() for s in re.split(r"\.\s+|\.$", body) if len(s.strip()) > 10]


def _extract_supporting(trace: Trace) -> str:
    """Concatenate findings/text/result fields from researcher/executor/retriever steps."""
    parts = []
    for s in trace.trajectory:
        if s.node in ("researcher", "executor", "retriever"):
            for key in ("findings", "text", "result"):
                if key in s.output:
                    parts.append(str(s.output[key]))
    return " ".join(parts).lower()


def groundedness(trace: Trace, threshold: float = 0.5) -> dict:
    """Rule-based: ≥`threshold` of claim's content words (≥4 chars, stopwords filtered)
    appear in supporting content."""
    claims = _extract_claims(trace.final_answer)
    if not claims:
        return {"grounded_fraction": 1.0, "total_claims": 0, "grounded_count": 0}
    content = _extract_supporting(trace)
    grounded = 0
    for c in claims:
        words = [w for w in re.findall(r"\w{4,}", c.lower()) if w not in _STOPWORDS]
        if not words or sum(1 for w in words if w in content) / len(words) >= threshold:
            grounded += 1
    return {"grounded_fraction": grounded / len(claims),
            "total_claims": len(claims), "grounded_count": grounded}


## Harness

`run_harness(trace_set)` builds the per-trace DataFrame. Aggregate-vs-slice and per-agent breakdown follow.

In [ ]:
def run_harness(trace_set: list[Trace]) -> pd.DataFrame:
    rows = []
    for t in trace_set:
        cp = citation_preservation(t)
        g = groundedness(t)
        rows.append({
            "trace_id": t.id, "source_lab": t.source_lab, "category": t.category,
            "handoff_success": handoff_success_rate(t),
            "routing_accuracy": routing_accuracy(t),
            "plan_validity": plan_validity(t),
            "plan_coverage": plan_coverage(t),
            "citation_preservation": cp["preservation"],
            "hallucinated_citations": cp["hallucinated_count"],
            "groundedness": g["grounded_fraction"],
            "claim_count": g["total_claims"],
        })
    return pd.DataFrame(rows)


df = run_harness(traces)
print(f"Per-trace metrics — {df.shape[0]} rows × {df.shape[1]} columns:")
print(df.to_string(index=False))


**Sample output**:

```
Per-trace metrics — 15 rows × 11 columns
(shape (15, 11) — table omitted here for brevity; same structure as parent Step 7)
```

## Aggregate vs slice

Aggregate first to find what's off; slice by category to localize. Never publish an aggregate without its slice.

In [ ]:
print("Aggregate (across all traces):")
print(df[["handoff_success", "routing_accuracy", "citation_preservation",
          "groundedness"]].mean().round(3))

print("\nSliced by category:")
print(df.groupby("category")[
    ["handoff_success", "routing_accuracy", "citation_preservation", "groundedness"]
].mean().round(3))

rr = replan_rate(traces)
print("\nReplan rate (Lab 12 traces only):")
print(f"  {rr['with_replans']}/{rr['total_lab12']} traces replanned at least once "
      f"(rate {rr['rate']:.2f}, mean {rr['mean_replans']:.2f} replans/run)")


**Sample output**:

```
Aggregate (across all traces):
handoff_success          0.892
routing_accuracy         0.967
citation_preservation    0.811
groundedness             0.411

Sliced by category:
                handoff_success  routing_accuracy  citation_preservation  groundedness
category
citation_drift            1.000             1.000                  0.556         0.222
happy_path                1.000             0.900                  0.800         0.500
replan_needed             0.775             1.000                  1.000         0.500
step_cap_hit              0.778             1.000                  1.000         0.167
tool_failure              0.750             1.000                  0.750         0.750

Replan rate (Lab 12 traces only):
  3/5 traces replanned at least once (rate 0.60, mean 0.80 replans/run)
```

The aggregate `groundedness = 0.411` looks alarming until the slice clarifies — `tool_failure` runs ground at 0.75; the lower scores come from `step_cap_hit` (which includes principled refusals the rule-based metric can't distinguish from hallucination) and `citation_drift` (which includes the writer's fabricated claim). Both are categories built to surface exactly these signatures.

## Per-agent breakdown

Re-key the trajectory data by agent. Shows which agent contributes which failure signal — the diagnostic step after aggregate-and-slice surface that something is wrong.

In [ ]:
rows = []
for t in traces:
    for s in t.trajectory:
        rows.append({
            "node": s.node,
            "is_error": s.status not in ("ok", "step_cap"),
            "is_step_cap": s.status == "step_cap",
        })
signal_df = pd.DataFrame(rows)

agent_signals = signal_df.groupby("node").agg(
    total_steps=("node", "size"),
    error_rate=("is_error", "mean"),
    step_cap_rate=("is_step_cap", "mean"),
).round(3).sort_values("error_rate", ascending=False)
print(agent_signals)


**Sample output**:

```
             total_steps  error_rate  step_cap_rate
node
planner                9       0.444         0.111
researcher            10       0.100         0.100
executor              12       0.083         0.000
critic                 8       0.000         0.000
supervisor            35       0.000         0.000
synthesizer            5       0.000         0.000
writer                13       0.000         0.000
```

The planner's 44% error rate is expected (validator-rejection-and-retry); flat 0% across supervisor/writer/critic/synthesizer means those agents handle their inputs without faulting in the fixture set. In production you'd expect non-zero rates as load grows — this set tests specific failure modes, not all of them.

## Synthesis

The harness is one pure-Python module — seven pure functions, one DataFrame builder, a category slicer, a per-agent groupby. The full mechanism for trajectory-level evaluation, with zero framework dependencies.

The pieces that make this work and scale:

- **Pure functions for metrics.** Trivial to unit-test; trivial to reuse in production tooling (LangSmith, Phoenix, Vertex AI all accept arbitrary scoring callables).
- **Pydantic-validated trace contract.** A typo in a fixture fails the harness loudly, not silently. Trace shape evolution is intentional and tracked.
- **Aggregation-then-slice-then-per-agent.** Three views of the same data; each one localizes failures the previous one can't.
- **Hand-curated fixtures over synthesis.** The trace_set is the spec; metrics are the implementation. 15 carefully-built traces beat 1500 generated ones for diagnostic coverage.

The pieces that aren't here yet — for Path 06:

- **Online (live) evaluation.** Score traces as they arrive; alert when category-level metrics drop.
- **Agent-as-judge calibration.** The Zheng et al. (2023) biases (position, verbosity, self-enhancement) need periodic calibration against human ground truth.
- **Production-grade trace ingestion.** OpenTelemetry-based; LangSmith / Phoenix / Galileo / Vertex AI as the layer above this harness.
- **Multi-turn (threaded) evaluation.** Conversational agents need trajectory metrics across turns; LangChain's Oct 2025 release is the documented path.

✓ **Path 03 v1 reference solutions complete.** Every lab from 01-16 (excluding 04) has a solution. The path is structurally closed.
